# DATA LOADING PIPELINE (POLARS)

This notebook contains **data loading and preprocessing pipeline** using **Polars** for optimal performance

**Purpose:**
- Load and validate datasets
- Apply discovered preprocessing strategies
- Prepare data for modeling

**Datasets:**
- train_full: 5,337,414 rows, 94 columns
- test_full: 1,447,107 rows, 92 columns

**Key Preprocessing Steps:**
1. Load datasets with Polars
2. Apply missing value strategies (based on analysis)
3. Create missing flags and filled columns
4. Handle weight distribution issues
5. Export processed data for modeling

## 1.1 IMPORTS AND CONFIGURATION

Setting up environment with all necessary libraries and Polars configuration.

In [20]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter

# Polars configuration
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


## 1.2 DATA LOADING

Loading the full train and test datasets using Polars with performance timing.

In [21]:
# ============================================
# LOAD FULL DATASETS
# ============================================
print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

# Load full datasets using Polars
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
test_full = pl.read_parquet(full_test_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Test full: {test_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

print(f"\n✅ Full datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Test full: (1447107, 92)
Load time: 9.13 seconds

✅ Full datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


In [22]:
train_full[0,0]

'W2MW3G2L__J0G2B0KU__PZ9S1Z4V__25__89'

## 1.3 DATA WINSORIZING

In [23]:
# ============================================
# SYMMETRIC CLIPPING (0.003 / 0.997)
# ============================================

feature_cols = [c for c in train_full.columns if c.startswith("feature_")]

clip_stats = []

for col in feature_cols:
    # quantiles z TRAIN
    q_low = train_full.select(pl.col(col).quantile(0.003)).item()
    q_high = train_full.select(pl.col(col).quantile(0.997)).item()

    # ile wartości będzie clipped w TRAIN
    train_low = train_full.select((pl.col(col) < q_low).sum()).item()
    train_high = train_full.select((pl.col(col) > q_high).sum()).item()

    # ile wartości będzie clipped w TEST
    test_low = test_full.select((pl.col(col) < q_low).sum()).item()
    test_high = test_full.select((pl.col(col) > q_high).sum()).item()

    total_train = train_full.height
    total_test = test_full.height

    clip_stats.append({
        "feature": col,

        "train_low_clipped": train_low,
        "train_high_clipped": train_high,
        "train_low_%": 100 * train_low / total_train,
        "train_high_%": 100 * train_high / total_train,

        "test_low_clipped": test_low,
        "test_high_clipped": test_high,
        "test_low_%": 100 * test_low / total_test,
        "test_high_%": 100 * test_high / total_test
    })

    # APPLY CLIPPING
    train_full = train_full.with_columns(
        pl.col(col).clip(q_low, q_high)
    )

    test_full = test_full.with_columns(
        pl.col(col).clip(q_low, q_high)
    )

clip_df = pl.DataFrame(clip_stats)

display(clip_df.sort("train_high_clipped", descending=True))

feature,train_low_clipped,train_high_clipped,train_low_%,train_high_%,test_low_clipped,test_high_clipped,test_low_%,test_high_%
str,i64,i64,f64,f64,i64,i64,f64,f64
"""feature_b""",16012,16012,0.299995,0.299995,4263,4307,0.294588,0.297628
"""feature_c""",16012,16012,0.299995,0.299995,4338,4250,0.299771,0.293689
"""feature_d""",16012,16012,0.299995,0.299995,4369,4337,0.301913,0.299701
"""feature_e""",16012,16012,0.299995,0.299995,4433,4354,0.306335,0.300876
"""feature_f""",16012,16012,0.299995,0.299995,4351,4383,0.300669,0.30288
"""feature_g""",16012,16012,0.299995,0.299995,4330,4403,0.299218,0.304262
"""feature_ai""",16005,16010,0.299864,0.299958,5464,11419,0.377581,0.789092
"""feature_bd""",15971,16010,0.299227,0.299958,8,900,0.000553,0.062193
"""feature_bf""",16007,16010,0.299902,0.299958,28,10498,0.001935,0.725447


## 1.3 DATA SIZE CHECKING

Checking the size of variables to fit them to float32

In [24]:
# ============================================
# DATA SIZE BEFORE OPTIMIZATION
# ============================================

def get_size_mb(df):
    return df.estimated_size() / 1024 ** 2


train_size_before = get_size_mb(train_full)
test_size_before = get_size_mb(test_full)

print("=" * 60)
print("DATA SIZE BEFORE OPTIMIZATION")
print("=" * 60)
print(f"Train size BEFORE: {train_size_before:.2f} MB")
print(f"Test size BEFORE : {test_size_before:.2f} MB")

DATA SIZE BEFORE OPTIMIZATION
Train size BEFORE: 3940.17 MB
Test size BEFORE : 1045.30 MB


In [25]:
schema_train = pl.DataFrame({
    "column": list(train_full.schema.keys()),
    "dtype": [str(v) for v in train_full.schema.values()]
})

display(schema_train)
schema_test = pl.DataFrame({
    "column": list(test_full.schema.keys()),
    "dtype": [str(v) for v in test_full.schema.values()]
})
#display(schema_test)

stats_train = train_full.select(
    [
        pl.col(c).min().alias(f"{c}_min")
        for c in train_full.columns
        if train_full[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
    ] +
    [
        pl.col(c).max().alias(f"{c}_max")
        for c in train_full.columns
        if train_full[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
    ]
)

display(stats_train)
stats_test = test_full.select(
    [
        pl.col(c).min().alias(f"{c}_min")
        for c in test_full.columns
        if test_full[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
    ] +
    [
        pl.col(c).max().alias(f"{c}_max")
        for c in test_full.columns
        if test_full[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
    ]
)

#display(stats_test)


column,dtype
str,str
"""id""","""String"""
"""code""","""String"""
"""sub_code""","""String"""
"""sub_category""","""String"""
"""horizon""","""Int32"""
"""ts_index""","""Int32"""
"""feature_a""","""Int32"""
"""feature_b""","""Float64"""
"""feature_c""","""Float64"""


horizon_min,ts_index_min,feature_a_min,feature_b_min,feature_c_min,feature_d_min,feature_e_min,feature_f_min,feature_g_min,feature_h_min,feature_i_min,feature_j_min,feature_k_min,feature_l_min,feature_m_min,feature_n_min,feature_o_min,feature_p_min,feature_q_min,feature_r_min,feature_s_min,feature_t_min,feature_u_min,feature_v_min,feature_w_min,feature_x_min,feature_y_min,feature_z_min,feature_aa_min,feature_ab_min,feature_ac_min,feature_ad_min,feature_ae_min,feature_af_min,feature_ag_min,feature_ah_min,feature_ai_min,feature_aj_min,feature_ak_min,feature_al_min,feature_am_min,feature_an_min,feature_ao_min,feature_ap_min,feature_aq_min,feature_ar_min,feature_as_min,feature_at_min,feature_au_min,feature_av_min,…,feature_am_max,feature_an_max,feature_ao_max,feature_ap_max,feature_aq_max,feature_ar_max,feature_as_max,feature_at_max,feature_au_max,feature_av_max,feature_aw_max,feature_ax_max,feature_ay_max,feature_az_max,feature_ba_max,feature_bb_max,feature_bc_max,feature_bd_max,feature_be_max,feature_bf_max,feature_bg_max,feature_bh_max,feature_bi_max,feature_bj_max,feature_bk_max,feature_bl_max,feature_bm_max,feature_bn_max,feature_bo_max,feature_bp_max,feature_bq_max,feature_br_max,feature_bs_max,feature_bt_max,feature_bu_max,feature_bv_max,feature_bw_max,feature_bx_max,feature_by_max,feature_bz_max,feature_ca_max,feature_cb_max,feature_cc_max,feature_cd_max,feature_ce_max,feature_cf_max,feature_cg_max,feature_ch_max,y_target_max,weight_max
i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64
1,1,0,0.180835,0.180387,0.180794,0.180986,0.180739,0.180848,0.000097,0.000053,0.000001,0.000003,0.030737,0.036645,-7.650148,0.0,0.0,0.000031,0.000156,0.0,0.000401,0.000455,0.00161,-217.19543,-347.437518,-835.495501,-1121.722471,0.0,0.0,0.000644,0.000461,0.000012,0.000831,2.5607e-9,0.005962,0.020123,0.013637,0.000081,-56.658399,0.001041,0.002233,0.010223,0.002038,0.02004,0.018099,0.035278,0.0,0.0,0.0,…,239.496404,382.556443,545.034076,314.7788,7.19183,6.462429,5.762582,117253.500202,193087.560205,103296.846646,162540.76126,170058.885787,130256.850924,78.193501,2.6684e6,296528.312006,494451.857662,1.7316e7,6.8298e6,5.6195e6,0.31063,6.7794e6,991.996856,258283.958952,366370.814292,167.594347,11053.672449,9670.285831,10445.757362,25.418763,414.070545,79.955221,-0.109063,-0.085446,-0.05872,4.089786,7.010022,11.826232,-0.000015,-0.000014,-0.000014,-0.000011,-0.000005,-0.000044,0.598266,6.483222,5.5673,9,2314.411152,1.3912e13


In [26]:
threshold = 1e7

large_cols = []

for col in train_full.columns:
    if train_full[col].dtype in [pl.Float64, pl.Int64]:
        max_val = train_full[col].max()
        if abs(max_val) > threshold:
            large_cols.append((col, max_val))

print("Columns with large values:")
print(large_cols)

Columns with large values:
[('feature_bd', 17315630.6353183), ('weight', 13912217783333.135)]


2.1 TYPE OPTIMIZATION

In [27]:
# ============================================
# TYPE OPTIMIZATION
# ============================================

print("="*60)
print("TYPE OPTIMIZATION")
print("="*60)

# float features
float_cols = [c for c in train_full.columns if c.startswith("feature_")]

train_full = train_full.with_columns(
    [pl.col(c).cast(pl.Float32) for c in float_cols]
)

test_full = test_full.with_columns(
    [pl.col(c).cast(pl.Float32) for c in float_cols]
)

# target
train_full = train_full.with_columns(
    pl.col("y_target").cast(pl.Float32)
)

# integers
train_full = train_full.with_columns([
    pl.col("horizon").cast(pl.Int16),
    pl.col("feature_a").cast(pl.Int16)
])

test_full = test_full.with_columns([
    pl.col("horizon").cast(pl.Int16),
    pl.col("feature_a").cast(pl.Int16)
])

print("✅ Type optimization complete")
cat_cols = ["code", "sub_code", "sub_category"]

train_full = train_full.with_columns(
    [pl.col(c).cast(pl.Categorical) for c in cat_cols]
)

test_full = test_full.with_columns(
    [pl.col(c).cast(pl.Categorical) for c in cat_cols]
)

print("✅ Categorical conversion complete")

TYPE OPTIMIZATION
✅ Type optimization complete
✅ Categorical conversion complete


In [28]:
def get_size_mb(df):
    return df.estimated_size() / 1024**2

print("Train size MB:", get_size_mb(train_full))
print("Test size MB:", get_size_mb(test_full))

Train size MB: 2107.7126722335815
Test size MB: 553.9911031723022


In [29]:
# ============================================
# SAVE PROCESSED DATA
# ============================================

processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

train_path = processed_dir / "train_processed.parquet"
test_path = processed_dir / "test_processed.parquet"

train_full.write_parquet(train_path)
test_full.write_parquet(test_path)

print("✅ Data saved")
print(train_path)
print(test_path)

✅ Data saved
..\data\processed\train_processed.parquet
..\data\processed\test_processed.parquet


In [19]:
train_full.select([
    pl.col(c).skew().alias(c)
    for c in feature_cols
])

feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,feature_g,feature_h,feature_i,feature_j,feature_k,feature_l,feature_m,feature_n,feature_o,feature_p,feature_q,feature_r,feature_s,feature_t,feature_u,feature_v,feature_w,feature_x,feature_y,feature_z,feature_aa,feature_ab,feature_ac,feature_ad,feature_ae,feature_af,feature_ag,feature_ah,feature_ai,feature_aj,feature_ak,feature_al,feature_am,feature_an,feature_ao,feature_ap,feature_aq,feature_ar,feature_as,feature_at,feature_au,feature_av,feature_aw,feature_ax,feature_ay,feature_az,feature_ba,feature_bb,feature_bc,feature_bd,feature_be,feature_bf,feature_bg,feature_bh,feature_bi,feature_bj,feature_bk,feature_bl,feature_bm,feature_bn,feature_bo,feature_bp,feature_bq,feature_br,feature_bs,feature_bt,feature_bu,feature_bv,feature_bw,feature_bx,feature_by,feature_bz,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.254473,-0.000531,0.000267,-0.000693,0.000821,0.000103,-0.000071,1.430075,1.208269,3.164309,1.731933,1.238398,0.816431,0.43222,8.209332,4.346589,5.481236,7.368658,6.37375,3.340017,5.227498,4.332879,-0.469421,-1.668905,-2.25216,-3.035923,5.095553,4.587408,4.126251,2.876938,6.267499,5.567379,8.595346,1.74909,0.712012,2.040932,3.616902,0.477636,3.904426,3.107154,3.687701,3.8391,0.924835,1.282995,1.043696,6.585084,5.813372,4.313008,5.089128,4.308861,6.009581,4.913928,9.908758,7.57069,6.206251,4.982894,3.28799,3.592104,2.64561,3.971727,11.467943,5.761092,4.962983,4.894402,4.457832,3.749244,5.135158,3.955064,6.057342,5.229791,-1.331268,-0.568707,0.242485,1.123389,0.911169,0.283692,-6.334991,-6.292277,-4.969699,-4.902393,-4.524224,-5.46518,-1.230089,0.839296,1.146129,1.205196


In [31]:
train_full.select(feature_cols).dtypes[:10]

[Int16,
 Float32,
 Float32,
 Float32,
 Float32,
 Float32,
 Float32,
 Float32,
 Float32,
 Float32]

## 2.0 MISSING VALUE HANDLING PIPELINE

Apply discovered preprocessing strategies based on comprehensive analysis.

In [ ]:
# ============================================
# MISSING VALUE HANDLING PIPELINE
# ============================================
print("🔧 APPLYING MISSING VALUE STRATEGIES")

# Priority columns based on analysis
priority_mcmc_dw = ['feature_bz', 'feature_aw', 'feature_cc']  # Δw impact
priority_mcmc_dy = ['feature_az', 'feature_bl', 'feature_l', 'feature_m']  # Δy impact
priority_test_38 = ['feature_w', 'feature_x', 'feature_y', 'feature_z']  # Test 38% missing
priority_core = ['feature_at', 'feature_by', 'feature_ay', 'feature_cd', 'feature_ce', 'feature_cf', 'feature_al']  # >5% missing

all_priority = priority_mcmc_dw + priority_mcmc_dy + priority_test_38 + priority_core

print(f"Priority 1 (MCMC Δw): {priority_mcmc_dw}")
print(f"Priority 2 (MCMC Δy): {priority_mcmc_dy}")
print(f"Priority 3 (TEST 38%): {priority_test_38}")
print(f"Priority 4 (CORE): {priority_core}")

print(f"\n⚡ Processing {len(all_priority)} priority columns...")

# Apply pipeline to all priority columns
for col in all_priority:
    # 1. Create missing flag
    train_full = train_full.with_columns([
        (pl.col(col).is_null()).cast(pl.Float32).alias(f'{col}_missing')
    ])
    
    # 2. Forward fill (time-series safe)
    train_full = train_full.with_columns([
        pl.col(col).fill_null(strategy="forward").alias(f'{col}_filled')
    ])
    
    print(f"✅ {col}: missing_flag + forward_fill")

print(f"\n📊 Results:")
print(f"   Original shape: {train_full.shape}")
print(f"   New shape: {train_full.shape}")
print(f"   Added columns: {len(all_priority) * 2} ({len(all_priority)} flags + {len(all_priority)} filled)")

🔧 APPLYING MISSING VALUE STRATEGIES
Priority 1 (MCMC Δw): ['feature_bz', 'feature_aw', 'feature_cc']
Priority 2 (MCMC Δy): ['feature_az', 'feature_bl', 'feature_l', 'feature_m']
Priority 3 (TEST 38%): ['feature_w', 'feature_x', 'feature_y', 'feature_z']
Priority 4 (CORE): ['feature_at', 'feature_by', 'feature_ay', 'feature_cd', 'feature_ce', 'feature_cf', 'feature_al']

⚡ Processing 18 priority columns...
✅ feature_bz: missing_flag + forward_fill
✅ feature_aw: missing_flag + forward_fill
✅ feature_cc: missing_flag + forward_fill
✅ feature_az: missing_flag + forward_fill
✅ feature_bl: missing_flag + forward_fill
✅ feature_l: missing_flag + forward_fill
✅ feature_m: missing_flag + forward_fill
✅ feature_w: missing_flag + forward_fill
✅ feature_x: missing_flag + forward_fill
✅ feature_y: missing_flag + forward_fill
✅ feature_z: missing_flag + forward_fill
✅ feature_at: missing_flag + forward_fill
✅ feature_by: missing_flag + forward_fill
✅ feature_ay: missing_flag + forward_fill
✅ feature

In [ ]:
# ============================================
# APPLY SAME PIPELINE TO TEST DATA
# ============================================
print("🔧 APPLYING SAME PIPELINE TO TEST DATA...")

print(f"⚡ Processing {len(all_priority)} priority columns...")

# Apply same pipeline to test data
for col in all_priority:
    # 1. Create missing flag
    test_full = test_full.with_columns([
        (pl.col(col).is_null()).cast(pl.Float32).alias(f'{col}_missing')
    ])
    
    # 2. Forward fill (time-series safe)
    test_full = test_full.with_columns([
        pl.col(col).fill_null(strategy="forward").alias(f'{col}_filled')
    ])
    
    print(f"✅ {col}: missing_flag + forward_fill")

print(f"\n📊 Test Results:")
print(f"   Original shape: {test_full.shape}")
print(f"   New shape: {test_full.shape}")
print(f"   Added columns: {len(all_priority) * 2} ({len(all_priority)} flags + {len(all_priority)} filled)")

🔧 APPLYING SAME PIPELINE TO TEST DATA...
⚡ Processing 18 priority columns...
✅ feature_bz: missing_flag + forward_fill
✅ feature_aw: missing_flag + forward_fill
✅ feature_cc: missing_flag + forward_fill
✅ feature_az: missing_flag + forward_fill
✅ feature_bl: missing_flag + forward_fill
✅ feature_l: missing_flag + forward_fill
✅ feature_m: missing_flag + forward_fill
✅ feature_w: missing_flag + forward_fill
✅ feature_x: missing_flag + forward_fill
✅ feature_y: missing_flag + forward_fill
✅ feature_z: missing_flag + forward_fill
✅ feature_at: missing_flag + forward_fill
✅ feature_by: missing_flag + forward_fill
✅ feature_ay: missing_flag + forward_fill
✅ feature_cd: missing_flag + forward_fill
✅ feature_ce: missing_flag + forward_fill
✅ feature_cf: missing_flag + forward_fill
✅ feature_al: missing_flag + forward_fill

📊 Test Results:
   Original shape: (1447107, 92)
   New shape: (1447107, 128)
   Added columns: 36 (18 flags + 18 filled)


## 3.0 WEIGHT DISTRIBUTION HANDLING

Address the extreme weight distribution skew (0.182% rows >1e9 control 99.9% LB).

In [ ]:
# ============================================
# WEIGHT DISTRIBUTION HANDLING
# ============================================
print("⚖️ WEIGHT DISTRIBUTION ANALYSIS")

# Analyze weight distribution
weight_stats = train_full.select([
    pl.col('weight').eq(0).sum().alias('weight_zeros'),
    pl.col('weight').gt(1e9).sum().alias('weight_high'),
    pl.col('weight').max().alias('max_weight'),
    pl.col('weight').mean().alias('mean_weight')
])

zeros = weight_stats[0, 'weight_zeros']
high = weight_stats[0, 'weight_high']
max_w = weight_stats[0, 'max_weight']
mean_w = weight_stats[0, 'mean_weight']

print(f"   Weight zeros: {zeros:,} ({zeros/len(train_full)*100:.3f}%)")
print(f"   Weight > 1e9: {high:,} ({high/len(train_full)*100:.3f}%)")
print(f"   Max weight: {max_w:,.2f}")
print(f"   Mean weight: {mean_w:,.2f}")

print("\n🔧 Creating weight features...")

# Create weight-related features
train_full = train_full.with_columns([
    pl.col('weight').log10().alias('weight_log10'),
    pl.col('weight').eq(0).cast(pl.Float32).alias('weight_is_zero'),
    pl.col('weight').gt(1e9).cast(pl.Float32).alias('weight_is_high'),
    pl.when(pl.col('weight').eq(0))
     .then(pl.lit('zero'))
     .when(pl.col('weight').gt(1e9))
     .then(pl.lit('high'))
     .otherwise(pl.lit('normal'))
     .alias('weight_category')
])

print("✅ weight_log10: Log10 transformation")
print("✅ weight_is_zero: Zero weight flag")
print("✅ weight_is_high: High weight (>1e9) flag")
print("✅ weight_category: Weight categorization")

# Show weight categories
weight_cats = train_full['weight_category'].value_counts()
print(f"\n📊 Weight categories:")
for cat in weight_cats['weight_category']:
    count = weight_cats.filter(pl.col('weight_category') == cat)['count'].item()
    print(f"   {cat}: {count:,} rows ({count/len(train_full)*100:.3f}%)")

⚖️ WEIGHT DISTRIBUTION ANALYSIS
   Weight zeros: 5,104 (0.096%)
   Weight > 1e9: 9,694 (0.182%)
   Max weight: 15,675,414,349.49
   Mean weight: 13,781,387.75

🔧 Creating weight features...
✅ weight_log10: Log10 transformation
✅ weight_is_zero: Zero weight flag
✅ weight_is_high: High weight (>1e9) flag
✅ weight_category: Weight categorization

📊 Weight categories:
   zero: 5,104 rows (0.096%)
   normal: 5,322,616 rows (99.722%)
   high: 9,694 rows (0.182%)


In [ ]:
# ============================================
# APPLY WEIGHT FEATURES TO TEST DATA
# ============================================
print("🔧 APPLYING WEIGHT FEATURES TO TEST DATA...")

# Apply same weight features to test data
test_full = test_full.with_columns([
    pl.col('weight').log10().alias('weight_log10'),
    pl.col('weight').eq(0).cast(pl.Float32).alias('weight_is_zero'),
    pl.col('weight').gt(1e9).cast(pl.Float32).alias('weight_is_high'),
    pl.when(pl.col('weight').eq(0))
     .then(pl.lit('zero'))
     .when(pl.col('weight').gt(1e9))
     .then(pl.lit('high'))
     .otherwise(pl.lit('normal'))
     .alias('weight_category')
])

print("✅ weight_log10: Log10 transformation")
print("✅ weight_is_zero: Zero weight flag")
print("✅ weight_is_high: High weight (>1e9) flag")
print("✅ weight_category: Weight categorization")

# Show test weight categories
test_weight_cats = test_full['weight_category'].value_counts()
print(f"\n📊 Test weight categories:")
for cat in test_weight_cats['weight_category']:
    count = test_weight_cats.filter(pl.col('weight_category') == cat)['count'].item()
    print(f"   {cat}: {count:,} rows ({count/len(test_full)*100:.3f}%)")

🔧 APPLYING WEIGHT FEATURES TO TEST DATA...
✅ weight_log10: Log10 transformation
✅ weight_is_zero: Zero weight flag
✅ weight_is_high: High weight (>1e9) flag
✅ weight_category: Weight categorization

📊 Test weight categories:
   zero: 1,376 rows (0.095%)
   normal: 1,445,731 rows (99.905%)
   high: 0 rows (0.000%)


## 4.0 FINAL DATA VALIDATION

Validate the processed datasets and save for modeling.

In [ ]:
# ============================================
# FINAL DATA VALIDATION
# ============================================
print("📊 FINAL DATASET SUMMARY")
print("="*60)

# Calculate added columns
added_cols = len(all_priority) * 2 + 5  # flags + filled + weight features

print("\nTRAIN DATASET:")
print(f"   Shape: ({train_full.shape[0]:,}, {train_full.shape[1]})")
print(f"   Original columns: 94")
print(f"   Added columns: {added_cols}")
print(f"   Missing flags: {len(all_priority)}")
print(f"   Filled columns: {len(all_priority)}")
print(f"   Weight features: 5")

print("\nTEST DATASET:")
print(f"   Shape: ({test_full.shape[0]:,}, {test_full.shape[1]})")
print(f"   Original columns: 92")
print(f"   Added columns: {added_cols}")
print(f"   Missing flags: {len(all_priority)}")
print(f"   Filled columns: {len(all_priority)}")
print(f"   Weight features: 5")

print("\n✅ Data processing complete!")
print("   Ready for modeling with LightGBM/XGBoost")

📊 FINAL DATASET SUMMARY

TRAIN DATASET:
   Shape: (5,337,414, 135)
   Original columns: 94
   Added columns: 41
   Missing flags: 18
   Filled columns: 18
   Weight features: 5

TEST DATASET:
   Shape: (1,447,107, 133)
   Original columns: 92
   Added columns: 41
   Missing flags: 18
   Filled columns: 18
   Weight features: 5

✅ Data processing complete!
   Ready for modeling with LightGBM/XGBoost


## 5.0 SAVE PROCESSED DATA

Save the processed datasets for modeling.

In [ ]:
# ============================================
# SAVE PROCESSED DATA
# ============================================
print("💾 SAVING PROCESSED DATA...")

# Create processed data directory
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

# Save processed datasets
train_path = processed_dir / "train_processed.parquet"
test_path = processed_dir / "test_processed.parquet"

print("⚡ Saving train_processed.parquet...")
train_full.write_parquet(train_path)
print(f"✅ Train saved: {train_path}")

print("⚡ Saving test_processed.parquet...")
test_full.write_parquet(test_path)
print(f"✅ Test saved: {test_path}")

print(f"\n📁 Files created:")
print(f"   train_processed.parquet: {train_full.shape[0]:,} rows × {train_full.shape[1]} columns")
print(f"   test_processed.parquet: {test_full.shape[0]:,} rows × {test_full.shape[1]} columns")

print("\n🎉 PIPELINE COMPLETE! Ready for modeling!")

💾 SAVING PROCESSED DATA...
⚡ Saving train_processed.parquet...
✅ Train saved: data/processed/train_processed.parquet
⚡ Saving test_processed.parquet...
✅ Test saved: data/processed/test_processed.parquet

📁 Files created:
   train_processed.parquet: 5,337,414 rows × 135 columns
   test_processed.parquet: 1,447,107 rows × 133 columns

🎉 PIPELINE COMPLETE! Ready for modeling!
